# Hugging Face & Transformers Library

In this notebook, you'll learn how to leverage the power of pre-trained models and the Hugging Face ecosystem to build state-of-the-art AI applications.

---

## Important: Enable GPU Runtime

This notebook runs models locally, including a 4-billion parameter language model. To avoid long wait times, you need a GPU runtime.

In Google Colab, go to **Runtime > Change runtime type** and select **T4 GPU** before running any code.

## What is Transfer Learning?

**Transfer learning** is the practice of taking a model that has already been trained on a large dataset and adapting it to your specific task. Instead of training a model from scratch (which requires massive amounts of data, compute power, and time), you leverage knowledge the model has already learned.



Language models like GPT, BERT, and others are trained on billions of words from the internet. This training:
- Costs millions of dollars in compute resources
- Takes weeks or months on specialized hardware
- Requires massive datasets (hundreds of gigabytes of text)

With transfer learning, you can use these pre-trained models **for free** and adapt them to your needs in minutes or hours instead of weeks! **Less training data** is needed since the pre-trained model brings its own knowledge. **Performance** is typically better because you are building on a model trained on massive datasets. And **the solutions are proven**, tested by millions of users across a wide range of tasks.






---

## Welcome to Hugging Face 🤗



**[Hugging Face](https://huggingface.co/)** is a company and platform that has become the "GitHub of machine learning." Founded in 2016 as a chatbot company, Hugging Face began focusing in 2018 on making state-of-the-art AI models available to everyone, regardless of budget or technical background.                     


It has become the go-to platform for the ML community for several reasons:
- Free and open-source tools
- Largest collection of pre-trained models (500,000+ models)
- Easy-to-use APIs that abstract away complexity
- Active community sharing models, datasets, and knowledge
- Excellent documentation and tutorials
- Works with all major ML frameworks (PyTorch, TensorFlow, JAX)

Today, Hugging Face is used by over 1 million developers and companies like Google, Microsoft, Meta, and Amazon.

---


### The Hugging Face Hub

The Hugging Face Hub is the central platform where everything lives. Let's take a look at its components:

1. **The Model Hub** hosts over 500,000 pre-trained models for every task imaginable (text generation, classification, translation, image generation, speech recognition, and more). You can filter models by task, language, license, and performance.

2. **The Datasets Hub** offers 100,000+ ready to use datasets (text, image, audio multimodal datasets) with automatic preprocessing and loading.

3. **Spaces** lets you host and share ML applications as web demos built with Gradio or Streamlit, so you can try models interactively before writing any code.

The platform also has comprehensive [documentation](https://huggingface.co/docs) with guides, API references, and community forums.






---

### Understanding Model Cards

Every model has a **Model Card** that describes it in detail. Before using any model in your project, you should read it to make sure the model fits your needs. Model Cards contain:

1. **Model Description**: What the model does and how it was trained
2. **Intended Use**: What tasks it's designed for
3. **Training Data**: What data was used (important for understanding biases)
4. **Performance Metrics**: How well it performs on benchmarks
5. **Limitations**: Known weaknesses and edge cases
6. **Bias and Fairness**: Potential biases in the model
7. **How to Use**: Code examples and usage instructions
8. **License**: Legal terms for usage

---


You might now wonder: *"How does Hugging Face make money if everything is free?"*

Everything we use in this tutorial is part of the **free tier**, which includes access to all open-source models, the Transformers library, public hosting, and community support. Hugging Face also offers a Pro plan ($9/month) with private hosting and higher compute limits, and Enterprise solutions with hosted inference endpoints, automated training, and dedicated support. The free tier is incredibly generous and sufficient for learning, prototyping, and many production use cases.




---

## Getting Started

### Creating a Hugging Face Account

To get the most out of Hugging Face, you'll want to create a free account:

1. Go to [huggingface.co](https://huggingface.co)
2. Click the **Sign Up** button in the top right
3. Enter your email address and create a password
4. Verify your email address (check your inbox)
5. Complete your profile (optional but recommended)

You can use most features without an account, but having one allows you to save models, create Spaces, and access private resources.

### Access Tokens

An **access token** is a secret key that authenticates your code when accessing Hugging Face resources. You need one for downloading gated models, uploading content, or accessing private repositories. For this tutorial, most models work without a token, but it is good practice to have one.

To create a token:
1. Log in to [huggingface.co](https://huggingface.co)
2. Click your profile picture > **Settings** > **Access Tokens**
3. Click **New token**, give it a name, select **Read** access
4. Click **Generate token** and copy it immediately

Store tokens securely using Colab Secrets or environment variables. Never commit tokens to git repositories or hardcode them in shared code.

### Install Dependencies

We need several libraries for this tutorial:
- **transformers** - The main library for using pre-trained models
- **datasets** - Easy access to datasets from the Hub
- **torch** - PyTorch deep learning framework (backend)
- **accelerate** - Simplifies running models on different hardware
- **sentencepiece** - Tokenizer used by many models

In [ ]:
# Install required libraries
!pip install -q transformers==5.0.0 tokenizers==0.22.2 datasets==4.0.0 accelerate==1.13.0 sentencepiece==0.2.1 protobuf==5.29.6

print("✅ All dependencies installed!")

### Setting Up Authentication (Optional)

If you have a Hugging Face access token, you can configure it here using Colab Secrets. This is optional for this tutorial since most models we use do not require authentication.

1. Click the 🔑 icon in the left sidebar
2. Click "Add new secret"
3. Name: `HF_TOKEN`
4. Value: Your Hugging Face token
5. Enable notebook access

In [ ]:
# Try to load Hugging Face token from Colab secrets
# This is optional - skip if you don't have a token

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ Hugging Face token loaded from Colab secrets")

    # Log in to Hugging Face
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ Successfully authenticated with Hugging Face")

except:
    print("ℹ️ No token found - continuing without authentication")
    print("   This is fine! Most models don't require authentication.")
    print("   You only need a token for gated models or uploading content.")

Perfect! You're now set up and ready to start using Hugging Face. Let's explore the Transformers library!

---

## The Transformers Library: High-Level APIs



The `transformers` library is designed with a clear philosophy: **Make state-of-the-art AI accessible to everyone, regardless of their expertise level.** To achieve this, the library provides **three levels of abstraction**:

#### Level 1: Pipelines (Easiest)
- **Who it's for**: Beginners, rapid prototyping, quick solutions
- **What you get**: One-line solutions for common tasks
- **Trade-off**: Less control, but incredibly easy
- **Example**: `classifier("I love this product!")` → `positive`

#### Level 2: Auto Classes (Balanced)
- **Who it's for**: Intermediate users who need more control
- **What you get**: Access to specific models and tokenizers
- **Trade-off**: More control, but requires understanding of tokenization
- **Example**: Choose exactly which model, customize preprocessing

#### Level 3: Raw Models (Most Control)
- **Who it's for**: Advanced users, researchers, custom training
- **What you get**: Complete control over every aspect
- **Trade-off**: Most flexibility, but requires deep knowledge
- **Example**: Access raw model weights, modify architecture


💡 **Key Point**: You can start with pipelines and gradually move to lower levels as your needs become more sophisticated. You don't need to learn everything at once!

---



### Pipelines: The Easiest Way to Use Models

**Pipelines** are the simplest way to use pre-trained models in Hugging Face. They handle everything for you:

- ✅ Downloading the right model
- ✅ Tokenizing input text
- ✅ Running inference
- ✅ Post-processing outputs
- ✅ Returning human-readable results

All in **one line of code**!

Pipelines are perfect for quick prototypes, standard tasks with default settings, and when you want results fast without worrying about the details. They are less suitable when you need fine-grained control over preprocessing, custom model modifications, or specific performance optimizations for production.


Hugging Face provides pipelines for dozens of tasks. Here are the most common:

**Text Tasks**:
- `text-classification` / `sentiment-analysis`: Classify text into categories
- `text-generation`: Generate continuing text
- `question-answering`: Answer questions based on context
- `summarization`: Create summaries of long texts
- `zero-shot-classification`: Classify without training examples
- `ner` (Named Entity Recognition): Extract entities (names, places, etc.)

**Audio Tasks**:
- `automatic-speech-recognition`: Transcribe speech to text
- `text-to-speech`: Convert text to audio
- `audio-classification`: Classify audio clips

**Image Tasks**:
- `image-classification`: Classify images
- `object-detection`: Detect objects in images
- `image-segmentation`: Segment images into regions
- `image-to-text`: Generate captions for images

**Multimodal Tasks**:
- `visual-question-answering`: Answer questions about images
- `document-question-answering`: Extract info from document images





Let's see pipelines in action!



### Sentiment Analysis Pipeline

Our first pipeline is **sentiment analysis**. It determines the emotional tone of text- is the message positive, negative, or neutral? This is one of the most common NLP tasks, used in practice for **monitoring customer reviews**, **tracking brand reputation** on social media, **prioritizing support tickets**, and **analyzing survey responses**.      



Let's create a sentiment analysis pipeline:

In [ ]:
from transformers import pipeline

# Create a sentiment analysis pipeline
# This automatically downloads and loads a pre-trained model
print("🔄 Loading sentiment analysis model...")
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("✅ Model loaded!\n")

# Test it with a simple example
result = sentiment_pipeline("I absolutely love this product! It's amazing!")
print(" Input: 'I absolutely love this product! It's amazing!'")
print(f" Result: {result}")

**What just happened?**

1. The pipeline downloaded a pre-trained sentiment model (stored in cache for reuse)
2. It tokenized your text (converted words to numbers the model understands)
3. It ran the model to get predictions
4. It formatted the output in a human-readable format

All in one line! Let's try more examples:

In [ ]:
# Test with different sentiments
test_texts = [
    "This is the worst experience I've ever had.",
    "The weather today is okay.",
    "I'm not sure how I feel about this.",
    "Incredible! Best purchase ever!",
    "It's fine, nothing special."
]

print("Testing multiple examples:\n")
for text in test_texts:
    result = sentiment_pipeline(text)[0]  # [0] because result is a list
    label = result['label']
    score = result['score']

    # Add emoji based on sentiment
    emoji = "😊" if label == "POSITIVE" else "😞"

    print(f"{emoji} '{text}'")
    print(f"   → {label} (confidence: {score:.1%})\n")

---

### Understanding the Model-Tokenizer Relationship

Now that you've seen a pipeline in action, let's understand what's happening under the hood. This is crucial for working with models effectively.

Neural networks work with numbers, not text. To process language, we need a two-part system:
- **The tokenizer** converts text into numbers (encoding)
- **The model** processes those numbers to produce predictions.

Every model comes with its own tokenizer that was trained specifically for it - you cannot mix and match, or the model will not understand the input.

The process works in three steps:

**Step 1: Text → Tokens**: the text is split into smaller pieces called tokens
```
"Hello, world!" → ["Hello", ",", "world", "!"]
```

**Step 2: Tokens → Token IDs**: each token is mapped to a numerical ID
```
["Hello", ",", "world", "!"] → [7592, 11, 995, 0]
```

**Step 3: Feed to Model**: IDs are fed into the model, which produces predictions
```
[7592, 11, 995, 0] → Model → Predictions
```

Let's peek inside the pipeline:

In [ ]:
# Access the tokenizer and model inside the pipeline
tokenizer = sentiment_pipeline.tokenizer
model = sentiment_pipeline.model

# Example text
text = "I love Hugging Face!"
print(f"📝 Original text: '{text}'\n")

# Step 1: Tokenize (text → tokens)
tokens = tokenizer.tokenize(text)
print(f" Tokens: {tokens}")

# Step 2: Convert to IDs (tokens → numbers)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f" Token IDs: {token_ids}\n")

# We can also do both steps at once
encoded = tokenizer(text)
print(f" Quick encoding (does both steps): {encoded['input_ids']}\n")

# And we can decode back to text
decoded = tokenizer.decode(encoded['input_ids'])
print(f"🔄 Decoded back to text: '{decoded}'")

#### Brief Overview of Tokenizer Types

Different models use different tokenization strategies:

**1. Word-based tokenization**
- Splits on spaces and punctuation
- Example: `"Hello world" → ["Hello", "world"]`
- Problem: Massive vocabulary, can't handle new words

**2. Character-based tokenization**
- Each character is a token
- Example: `"Hello" → ["H", "e", "l", "l", "o"]`
- Problem: Very long sequences, loses word meaning

**3. Subword tokenization** (Most common - used by BERT, GPT, etc.)
- Breaks words into meaningful pieces
- Example: `"unhappiness" → ["un", "happiness"]`
- Advantages: Manageable vocabulary, handles new words, preserves meaning

Common subword algorithms:
- **BPE (Byte-Pair Encoding)**: Used by GPT, Llama
- **WordPiece**: Used by BERT
- **SentencePiece**: Used by T5, ALBERT



### Sentiment Analysis with Real Data

Now let's use our sentiment pipeline on a real dataset from Hugging Face. We'll analyze movie reviews from the IMDb dataset.

First, let's load a small sample of the dataset:

In [ ]:
from datasets import load_dataset

# Load a small sample of movie reviews
print(" Loading IMDb movie review dataset...")
dataset = load_dataset("imdb", split="test[:10]")  # Load first 10 test examples
print(f"✅ Loaded {len(dataset)} reviews\n")

# Look at the first review
print(" Example review:")
print(f"Text: {dataset[0]['text'][:200]}...")  # Show first 200 characters
print(f"Actual label: {dataset[0]['label']} (0=negative, 1=positive)")

Now let's analyze these reviews with our sentiment pipeline:

In [ ]:
# Analyze sentiment for each review
print(" Analyzing movie reviews:\n")
print("=" * 80)

# Fix: Iterate correctly over the dataset
for i in range(min(5, len(dataset))):  # Analyze first 5
    # Get the review text and actual label
    review = dataset[i]  # Access each review individually
    text = review['text']
    actual_label = "POSITIVE" if review['label'] == 1 else "NEGATIVE"

    # Truncate long reviews (models have max length limits)
    text_preview = text[:250] + "..." if len(text) > 150 else text

    # Get prediction from pipeline
    prediction = sentiment_pipeline(text)[0]
    predicted_label = prediction['label']
    confidence = prediction['score']

    # Check if prediction matches actual label
    is_correct = actual_label == predicted_label
    status_emoji = "✅" if is_correct else "❌"

    print(f"\n{status_emoji} Review {i+1}:")
    print(f"Text: {text_preview}")
    print(f"Actual: {actual_label} | Predicted: {predicted_label} (confidence: {confidence:.1%})")

print("\n" + "=" * 80)

#### Customizing Parameters

Pipelines allow some customization. Let's explore the options:

In [ ]:
# You can specify which model to use
# Default is distilbert-base-uncased-finetuned-sst-2-english
# Let's try a different sentiment model

print(" Loading a different sentiment model...\n")

# Using a Twitter-specific sentiment model
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

twitter_sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

# This model has 3 labels: negative, neutral, positive
test_tweets = [
    "@HuggingFace is awesome! 🎉",
    "I don't really care either way.",
    "This is terrible. Not happy at all."
]

print(" Testing with Twitter-trained model:\n")
for tweet in test_tweets:
    result = twitter_sentiment(tweet)[0]
    print(f"Tweet: {tweet}")
    print(f"Sentiment: {result['label']} (confidence: {result['score']:.1%})\n")

#### Understanding Outputs

Pipeline outputs are dictionaries with consistent structure:

```python
[
    {
        'label': 'POSITIVE',    # The predicted class
        'score': 0.9998         # Confidence (probability between 0-1)
    }
]
```

**Key points**:
- Output is always a **list** (even for single inputs)
- `label`: The predicted category
- `score`: Confidence level (closer to 1.0 = more confident)

**When to trust predictions**:
- `score > 0.9`: Very confident, likely accurate
- `score 0.7-0.9`: Confident, usually reliable
- `score 0.5-0.7`: Uncertain, consider edge case
- `score < 0.5`: Not confident (shouldn't happen with 2 classes)

💡 **Key Takeaway**: Always check the confidence score, not just the label. Low confidence predictions may need human review.

---

### Quick Examples of Other Pipelines

Before we move deeper, let's quickly see a few other pipeline tasks in action:

#### Text Generation

In [ ]:
# Text generation - the model receives an unfinished prompt and generates a continuation
from transformers import pipeline, GenerationConfig

print(" Text Generation Pipeline\n")

generator = pipeline("text-generation", model="gpt2")
prompt = "Artificial intelligence will"
result = generator(
    prompt,
    generation_config=GenerationConfig(
        max_new_tokens=50,
        num_return_sequences=2,
        temperature=0.7,
        do_sample=True,
        pad_token_id=generator.tokenizer.eos_token_id
    )
)

print(f" Prompt: '{prompt}'\n")
for i, generation in enumerate(result, 1):
    print(f"{i}. {generation['generated_text']}\n")

#### Zero-Shot Classification

This is one of the most powerful pipelines - it classifies text into categories **you define**, without any training!

In [ ]:
# Zero-shot classification - classify without training examples!
print("Zero-Shot Classification Pipeline\n")

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define our custom categories
text = "I need to reset my password because I forgot it."
candidate_labels = ["technical support", "billing inquiry", "product feedback", "account issue"]

result = classifier(text, candidate_labels)

print(f"Text: '{text}'\n")
print("lassification results:")
for label, score in zip(result['labels'], result['scores']):
    bar = "█" * int(score * 20)  # Visual bar
    print(f"  {label:20s} {bar} {score:.1%}")

💡 **This is incredibly powerful!** You can create a classifier for any categories you want without training any model!

---

## Running Models Locally

You might wonder why you would run a model on your own machine when APIs like OpenAI are so convenient. There are strong reasons:

**Privacy and Security**
- Your data never leaves your machine
- No third-party can access your queries
- Critical for sensitive data (medical, legal, financial)

**Cost Efficiency**
- No per-token charges
- Pay once for hardware, use unlimited times
- Better for high-volume applications

**Offline Capability**
- Works without internet connection
- No dependency on API availability
- Guaranteed uptime

**Understanding**
- See exactly how models work
- Experiment with parameters

**Customization**
- Full control over generation parameters
- Can modify model architecture
- Fine-tune for specific tasks

On the other hand there are some disadvantages as well. You need powerful hardware (a GPU with 8GB+ VRAM is recommended), models take up significant storage space (2-20GB per model), and smaller open-source models may not match the quality of the latest commercial offerings.

Many companies use a hybrid approach - APIs for production quality, local models for development and experimentation.

---

### Model Selection: Qwen3-4B-Instruct

For this demo, we'll use **Qwen3-4B-Instruct-2507**, an excellent model that balances quality and resource requirements.



**Technical Specifications**:
- **Size**: 4 billion parameters
- **Memory**: ~8GB RAM minimum
- **Context**: 32K tokens (very long conversations)
- **Training**: High-quality multilingual data
- **Special**: Instruction-tuned for following commands
- **Version**: Latest 2507 release

This model runs on free Colab with a T4 GPU, produces excellent quality for its size, and supports multiple languages. It is released under the Apache 2.0 license, which means you can use it for commercial projects as well.


---

### Downloading and Loading the Model

Let's download and load Qwen3-4B-Instruct-2507. This will take a few minutes on the first run.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   Note: Running on CPU will be slower")

print("\n Loading Qwen3-4B-Instruct-2507 model...")
print("   This will download ~8GB on first run (cached for future use)")
print("   Please wait 2-3 minutes...\n")

# Model name on Hugging Face Hub
model_name = "Qwen/Qwen3-4B-Instruct-2507"

# Load tokenizer
print(" Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("   ✅ Tokenizer loaded")

# Load model
print("\n Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # Use half precision to save memory
    device_map="auto",           # Automatically place on available device
    low_cpu_mem_usage=True       # Optimize memory usage
)
print("   ✅ Model loaded")

print("\n" + "="*60)
print(" Qwen3-4B-Instruct-2507 is ready to use!")
print("="*60)



Let's see where the model files are stored and what they contain:

In [ ]:
from pathlib import Path
import os

# Models are cached in the Hugging Face cache directory
cache_dir = Path.home() / ".cache" / "huggingface" / "hub"

print(" Model Storage Information\n")
print(f"Cache directory: {cache_dir}")
print("\nHow caching works:")
print("  1. First time: Model downloads to cache (~6GB)")
print("  2. Subsequent times: Loads instantly from cache")
print("  3. Shared across projects: One download, use everywhere")
print("\n💡 Tip: To save space, you can clear cache with:")
print("   !rm -rf ~/.cache/huggingface/hub")
print("   (but then you'll need to re-download models)")

> Note: In Google Colab, the cache is temporary. When your runtime disconnects or restarts, downloaded models are deleted and will need to be downloaded again next time. On your local machine, the cache is permanent and shared across all your projects.

A model download contains several important files:

```
model_directory/
├── config.json              # Model architecture and settings
├── tokenizer.json           # Tokenizer vocabulary and rules
├── tokenizer_config.json    # Tokenizer settings
├── special_tokens_map.json  # Special tokens (BOS, EOS, etc.)
├── pytorch_model.bin        # Model weights (the big file!)
└── README.md                # Model card documentation
```

The largest file is `pytorch_model.bin` which contains the actual neural network weights. Everything else is configuration that tells the library how to load and use the model correctly.

### Exploring the Model in Depth

Now let's really understand how this model works.



Qwen3 is a **decoder-only transformer** model, similar to GPT. Here's what that means:


1. **Token Embeddings**: Converts token IDs to dense vectors
2. **Transformer Layers**: Stacked layers that process information
   - Self-attention: "Looks at" all previous tokens
   - Feed-forward: Processes each token independently
3. **Output Layer**: Predicts next token probabilities

**How it generates text**:
```
Input: "The cat sat on the"
  ↓ Tokenize
[2,1544,8718,525,356,464]
  ↓ Embed
[vector₁, vector₂, ...]
  ↓ Transformer layers
Process and attend to all tokens
  ↓ Output layer
Probabilities: {"mat": 0.6, "chair": 0.2, "floor": 0.15, ...}
  ↓ Sample
"mat" (most likely)
```

Let's see the model's structure:

In [ ]:
# Inspect model architecture
print("  Model Architecture\n")
print(f"Model class: {model.__class__.__name__}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"\nConfiguration:")
print(f"  Vocabulary size: {model.config.vocab_size:,}")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Number of layers: {model.config.num_hidden_layers}")
print(f"  Number of attention heads: {model.config.num_attention_heads}")
print(f"  Maximum position embeddings: {model.config.max_position_embeddings:,}")

print("\n💡 What these numbers mean:")
print("  - Parameters: More = more capable (but slower and needs more RAM)")
print("  - Vocabulary size: Number of unique tokens the model knows")
print("  - Hidden size: Dimension of internal representations")
print("  - Layers: More = deeper understanding (but slower)")
print("  - Max positions: Maximum input length (32K tokens = ~24K words)")



Let's explore tokenization in detail with real examples:

In [ ]:
print(" Tokenization Deep Dive\n")
print("="*70)

# Example 1: Simple sentence
text1 = "Hello, world!"
print(f"\n📝 Text: '{text1}'")

# Tokenize
tokens = tokenizer.tokenize(text1)
print(f"Tokens: {tokens}")

# Convert to IDs
token_ids = tokenizer.encode(text1, add_special_tokens=False)
print(f"Token IDs: {token_ids}")

# Decode back
decoded = tokenizer.decode(token_ids)
print(f"Decoded: '{decoded}'")

# Example 2: Subword tokenization
print("\n" + "="*70)
text2 = "unhappiness"
print(f"\n📝 Text: '{text2}'")
tokens2 = tokenizer.tokenize(text2)
print(f"Tokens: {tokens2}")
print("   Notice how 'unhappiness' breaks into meaningful parts!")

# Example 3: Handling unknown words
print("\n" + "="*70)
text3 = "supercalifragilisticexpialidocious"
print(f"\n📝 Text: '{text3}'")
tokens3 = tokenizer.tokenize(text3)
print(f"Tokens: {tokens3}")
print("   Even made-up words can be tokenized using subword units!")



Special tokens are important markers that give structure to the input:

In [ ]:
print(" Special Tokens\n")

# Show special tokens
print("Token          | Purpose")
print("-" * 50)
print(f"BOS (Beginning)| {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
print(f"EOS (End)      | {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD (Padding)  | {tokenizer.pad_token} (ID: {tokenizer.pad_token_id if tokenizer.pad_token else 'None'})")
print(f"UNK (Unknown)  | {tokenizer.unk_token} (ID: {tokenizer.unk_token_id})")

print("\n💡 What they do:")
print("  BOS: Marks the start of a sequence")
print("  EOS: Marks the end - tells model to stop generating")
print("  PAD: Fills shorter sequences in batches to same length")
print("  UNK: Represents truly unknown tokens (rare with subword tokenization)")

# Show encoding with special tokens
print("\n" + "="*70)
text = "Hello!"
print(f"\n Original: '{text}'")

ids_without = tokenizer.encode(text, add_special_tokens=False)
ids_with = tokenizer.encode(text, add_special_tokens=True)

print(f"\nWithout special tokens: {ids_without}")
print(f"With special tokens:    {ids_with}")
print("\nNotice the BOS token added at the beginning!")



Now let's peek at the model's vocabulary:

In [ ]:
# Get vocabulary
vocab = tokenizer.get_vocab()

print(f" Vocabulary Information\n")
print(f"Total vocabulary size: {len(vocab):,} tokens")

# Show some random tokens
import random
sample_tokens = random.sample(list(vocab.items()), 15)

print("\n Random sample of tokens:")
print("\nToken          | ID")
print("-" * 30)
for token, idx in sorted(sample_tokens, key=lambda x: x[1]):
    # Replace special characters for display
    display_token = token.replace('▁', '_').replace('\n', '\\n')
    print(f"{display_token:15s}| {idx}")



Now for the most important part: understanding **how the model generates text**:



1. **Start with prompt**: "The weather today is"
2. **Model predicts next token**: Calculates probabilities for every token in vocabulary
3. **Sample a token**: Choose one based on probabilities (e.g., "beautiful")
4. **Append to prompt**: "The weather today is beautiful"
5. **Repeat**: Use new prompt to predict next token
6. **Stop**: When model generates EOS token or max length reached

Let's see this step-by-step:

In [ ]:
import torch.nn.functional as F

print(" Step-by-Step Text Generation\n")
print("="*70)

# Start with a prompt
prompt = "Artificial intelligence is"
print(f" Prompt: '{prompt}'\n")

# Tokenize
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
print(f" Input IDs: {input_ids.tolist()[0]}")

# Generate one token at a time (manual generation loop)
print("\n Generating tokens one at a time:\n")

generated_ids = input_ids.clone()

for step in range(5):  # Generate 5 tokens
    # Get model predictions
    with torch.no_grad():
        outputs = model(generated_ids)
        # Get logits for the last token
        next_token_logits = outputs.logits[0, -1, :]

    # Convert logits to probabilities
    probs = F.softmax(next_token_logits, dim=-1)

    # Get top 5 most likely tokens
    top_probs, top_indices = torch.topk(probs, 5)

    print(f"Step {step + 1}:")
    print(f"  Current text: '{tokenizer.decode(generated_ids[0])}'")
    print(f"  Top 5 next token predictions:")
    for prob, idx in zip(top_probs, top_indices):
        token = tokenizer.decode([idx])
        print(f"    '{token}' - {prob.item():.1%}")

    # Sample the most likely token (greedy)
    next_token_id = top_indices[0].unsqueeze(0).unsqueeze(0)
    generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

    print(f"  ✅ Chose: '{tokenizer.decode([top_indices[0]])}'\n")

final_text = tokenizer.decode(generated_ids[0])
print("="*70)
print(f"\n Final generated text:\n'{final_text}'")

#### Sampling Strategies

There are different ways to choose the next token from the probability distribution:

**1. Greedy Decoding** always picks the most likely token - it is deterministic and coherent but tends to be repetitive. This works well for factual tasks like translation or answering specific questions where you want a single correct answer.  

**2. Sampling** picks randomly based on probabilities, which produces diverse and creative text but can sometimes lose coherence.

**3. Top-k Sampling** limits the choice to the k most likely tokens, which gives a good balance between diversity and coherence for general text generation.

**4. Top-p (Nucleus) Sampling** is similar but adapts to context - it considers tokens until their cumulative probability reaches p, so the model has more options when it is uncertain. This is the most popular strategy for high-quality generation and works well for tasks like chatbots and content creation.

These strategies can be combined with **temperature scaling**, which adjusts the probability distribution before sampling:
- **Low temperature (0.1-0.5)**: makes the model more focused and deterministic, which is useful for code generation or factual summaries
- **Medium temperature (0.7-1.0)**:  gives a balanced output suitable for most tasks
- **High temperature (1.5-2.0)**:  produces more creative and surprising text, which can be useful for brainstorming or creative writing

Let's compare these strategies:

In [ ]:
print(" Comparing Sampling Strategies\n")
print("="*70)

prompt = "Once upon a time, there was a"
print(f" Prompt: '{prompt}'\n")

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

# Strategy 1: Greedy (deterministic)
print("1️⃣ Greedy Decoding (always pick most likely):")
output_greedy = model.generate(
    input_ids,
    max_length=50,
    do_sample=False,  # Greedy
    pad_token_id=tokenizer.eos_token_id
)
print(f"   {tokenizer.decode(output_greedy[0], skip_special_tokens=True)}\n")

# Strategy 2: Temperature sampling (creative)
print("2️⃣ High Temperature (creative, diverse):")
output_temp = model.generate(
    input_ids,
    max_length=50,
    do_sample=True,
    temperature=1.5,  # High = more random
    pad_token_id=tokenizer.eos_token_id
)
print(f"   {tokenizer.decode(output_temp[0], skip_special_tokens=True)}\n")

# Strategy 3: Top-p sampling (balanced)
print("3️⃣ Top-p Sampling (balanced, high quality):")
output_topp = model.generate(
    input_ids,
    max_length=50,
    do_sample=True,
    top_p=0.9,  # Consider tokens until 90% probability mass
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)
print(f"   {tokenizer.decode(output_topp[0], skip_special_tokens=True)}\n")

# Strategy 4: Top-k sampling
print("4️⃣ Top-k Sampling (controlled diversity):")
output_topk = model.generate(
    input_ids,
    max_length=50,
    do_sample=True,
    top_k=50,  # Only consider top 50 tokens
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)
print(f"   {tokenizer.decode(output_topk[0], skip_special_tokens=True)}\n")

print("="*70)
print("\n💡 Notice how different strategies produce different outputs!")

Now that we know how the model chooses each next token, the remaining question is: *when does it stop?* Generation ends in one of three cases:


1. **EOS token generated**: Model decides it's done
2. **Max length reached**: Safety limit to prevent infinite generation
3. **Custom stopping criteria**: You can define custom rules

The most important parameters for controlling this are the following:
- `max_length`: Maximum total tokens (input + output)
- `max_new_tokens`: Maximum tokens to generate (clearer)
- `min_length`: Force minimum length (prevents premature stopping)


Let's wrap what we learned into a simple chatbot function. It takes a user message, formats it as a conversation, generates a response using top-p sampling, and keeps track of the conversation history so the model remembers previous messages:

In [ ]:
def chat_with_qwen(message, conversation_history="", max_length=200):
    """
    Simple chatbot using Qwen3-4B-Instruct-2507.

    Args:
        message: User's message
        conversation_history: Previous conversation (optional)
        max_length: Maximum response length

    Returns:
        tuple: (response, updated_conversation_history)
    """
    # Format the conversation
    prompt = f"{conversation_history}Human: {message}\nAssistant:"

    # Tokenize
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_length=input_ids.shape[1] + max_length,
            do_sample=True,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode the response
    full_response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Extract just the assistant's response
    response = full_response.split("Assistant:")[-1].strip()

    # Update conversation history
    updated_history = f"{conversation_history}Human: {message}\nAssistant: {response}\n"

    return response, updated_history

# Test the chatbot
print("Chatbot Demo\n")
print("="*70)

conversation = ""

# Turn 1
user_msg = "Hello! What can you help me with?"
print(f"👤 Human: {user_msg}")
response, conversation = chat_with_qwen(user_msg, conversation)
print(f"🤖 Assistant: {response}\n")

# Turn 2
user_msg = "Tell me a fun fact about space."
print(f"👤 Human: {user_msg}")
response, conversation = chat_with_qwen(user_msg, conversation)
print(f"🤖 Assistant: {response}\n")

# Turn 3
user_msg = "That's interesting! Tell me more."
print(f"👤 Human: {user_msg}")
response, conversation = chat_with_qwen(user_msg, conversation)
print(f"🤖 Assistant: {response}\n")

print("="*70)

The function above runs predefined messages. Now let's make it interactive - **you can type your own questions and have a real conversation with the model**:

In [ ]:
print("Interactive Chatbot")
print("="*70)
print("Chat with Qwen3-4B-Instruct! Type your messages and press Enter.")
print("Type 'quit' or 'exit' to end the conversation.")
print("="*70 + "\n")

conversation = ""

while True:
    # Get user input
    user_message = input("👤 You: ").strip()

    # Check for exit commands
    if user_message.lower() in ['quit', 'exit', 'bye']:
        print("\n🤖 Assistant: Goodbye! Have a great day!")
        break

    # Skip empty messages
    if not user_message:
        continue

    # Generate response
    print("🤖 Assistant: ", end="", flush=True)
    response, conversation = chat_with_qwen(user_message, conversation, max_length=150)
    print(response + "\n")

## Practice Exercise


Time to apply what you've learned! Your task is finding a model on Hugging Face for a specific task and use it.


1. **Choose a task** from the list:
   - Text summarization
   - Named Entity Recognition (NER)
   - Question Answering
   - Translation (any language pair)

2. **Find a model** on the Hugging Face Hub:
   - Go to [huggingface.co/models](https://huggingface.co/models)
   - Filter by your chosen task
   - Pick a model with good downloads/likes
   - Read the model card

3. **Implement it** using a pipeline:
   - Load the model with `pipeline()`
   - Test it with at least 3 different inputs
   - Print the results


💡 **Tips**:
- Start with popular models (high downloads)
- Read the model card carefully for usage examples
- Don't worry if it takes time to load - first run always downloads
- Experiment with different inputs to see strengths and weaknesses